# Lab | Data Cleaning and Formatting

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv


# Challenge 1: Data Cleaning and Formatting

## Exercise 1: Cleaning Column Names

To ensure consistency and ease of use, standardize the column names of the dataframe. Start by taking a first look at the dataframe and identifying any column names that need to be modified. Use appropriate naming conventions and make sure that column names are descriptive and informative.

*Hint*:
- *Column names should be in lower case*
- *White spaces in column names should be replaced by `_`*
- *`st` could be replaced for `state`*

### Solution — load the data and standardize names

First, import pandas and load the CSV. `head()` lets us confirm that the data loaded, while `columns` shows the original labels.

For consistent *snake_case* names, we strip accidental outer spaces, convert to lowercase, and replace spaces with underscores. Finally, `st` is renamed to the more descriptive `state`. Each operation is vectorized across the column-name index.


In [ ]:
import pandas as pd

data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(data_url)

display(df.head())
print("Original names:", df.columns.tolist())

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
df = df.rename(columns={"st": "state"})

print("Clean names:", df.columns.tolist())


## Exercise 2: Cleaning invalid Values

The dataset contains columns with inconsistent and incorrect values that could affect the accuracy of our analysis. Therefore, we need to clean these columns to ensure that they only contain valid data.

Note that this exercise will focus only on cleaning inconsistent values and will not involve handling null values (NaN or None).

*Hint*:
- *Gender column contains various inconsistent values such as "F", "M", "Femal", "Male", "female", which need to be standardized, for example, to "M" and "F".*
- *State abbreviations be can replaced with its full name, for example "AZ": "Arizona", "Cali": "California", "WA": "Washington"*
- *In education, "Bachelors" could be replaced by "Bachelor"*
- *In Customer Lifetime Value, delete the `%` character*
- *In vehicle class, "Sports Car", "Luxury SUV" and "Luxury Car" could be replaced by "Luxury"*

### Solution — standardize inconsistent values

Before changing anything, `unique()` reveals the exact variants. `replace()` then maps known variants to one canonical label. Missing values are intentionally left untouched in this exercise.

For customer lifetime value, `.str.replace()` removes `%` but leaves the column as text for now; conversion to numeric belongs in Exercise 3.


In [ ]:
for column in ["gender", "state", "education", "vehicle_class"]:
    print(f"{column}: {df[column].unique()}")

df["gender"] = df["gender"].replace({
    "Femal": "F", "female": "F", "Female": "F",
    "Male": "M", "male": "M"
})

df["state"] = df["state"].replace({
    "AZ": "Arizona", "Cali": "California", "WA": "Washington"
})

df["education"] = df["education"].replace({"Bachelors": "Bachelor"})

df["customer_lifetime_value"] = (
    df["customer_lifetime_value"].str.replace("%", "", regex=False)
)

df["vehicle_class"] = df["vehicle_class"].replace({
    "Sports Car": "Luxury",
    "Luxury SUV": "Luxury",
    "Luxury Car": "Luxury"
})

for column in ["gender", "state", "education", "vehicle_class"]:
    print(f"Clean {column}: {df[column].unique()}")


## Exercise 3: Formatting data types

The data types of many columns in the dataset appear to be incorrect. This could impact the accuracy of our analysis. To ensure accurate analysis, we need to correct the data types of these columns. Please update the data types of the columns as appropriate.

It is important to note that this exercise does not involve handling null values (NaN or None).

*Hint*:
- *Customer lifetime value should be numeric*
- *Number of open complaints has an incorrect format. Look at the different values it takes with `unique()` and take the middle value. As an example, 1/5/00 should be 5. Number of open complaints is a string - remember you can use `split()` to deal with it and take the number you need. Finally, since it should be numeric, cast the column to be in its proper type.*

### Solution — correct data types

`dtypes` shows pandas' current interpretation. Customer lifetime value is converted with `pd.to_numeric`; `errors="coerce"` changes malformed non-null entries into `NaN`, making them available for the missing-value strategy in the next exercise.

Complaint values such as `1/5/00` are encoded strings. Splitting at `/` produces `['1', '5', '00']`; `.str[1]` selects the middle value (`5`), which is then converted to a number.


In [ ]:
print("Before formatting:")
display(df.dtypes)
print("Complaint values:", df["number_of_open_complaints"].unique())

df["customer_lifetime_value"] = pd.to_numeric(
    df["customer_lifetime_value"], errors="coerce"
)

df["number_of_open_complaints"] = pd.to_numeric(
    df["number_of_open_complaints"].str.split("/").str[1],
    errors="coerce"
)

numeric_columns = ["income", "monthly_premium_auto", "total_claim_amount"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

print("After formatting:")
display(df.dtypes)


## Exercise 4: Dealing with Null values

Identify any columns with null or missing values. Identify how many null values each column has. You can use the `isnull()` function in pandas to find columns with null values.

Decide on a strategy for handling the null values. There are several options, including:

- Drop the rows or columns with null values
- Fill the null values with a specific value (such as the column mean or median for numerical variables, and mode for categorical variables)
- Fill the null values with the previous or next value in the column
- Fill the null values based on a more complex algorithm or model (note: we haven't covered this yet)

Implement your chosen strategy to handle the null values. You can use the `fillna()` function in pandas to fill null values or `dropna()` function to drop null values.

Verify that your strategy has successfully handled the null values. You can use the `isnull()` function again to check if there are still null values in the dataset.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling null values.

After formatting data types, as a last step, convert all the numeric variables to integers.

### Solution — handle missing values

We first count missing values with `isnull().sum()`.

Chosen strategy:

- Drop rows missing `customer`, because an insurance record without its identifying key cannot reliably be connected to a customer.
- Fill numeric gaps with the **median**, which is less sensitive to unusually large values than the mean.
- Fill categorical gaps with the **mode**, the most frequent valid label, so we retain the row without inventing a new category.
- Convert all numeric columns to integers last, as requested. This truncates decimals; in real financial work, retaining cents as decimals would usually be safer.

The final null count verifies the result.


In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum().rename("missing_count"))

df = df.dropna(subset=["customer"]).copy()

numeric_columns = df.select_dtypes(include="number").columns
categorical_columns = df.select_dtypes(exclude="number").columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

for column in categorical_columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].mode().iloc[0])

df[numeric_columns] = df[numeric_columns].astype(int)

print("Missing values after cleaning:")
display(df.isnull().sum().rename("missing_count"))
display(df.dtypes)


## Exercise 5: Dealing with duplicates

Use the `.duplicated()` method to identify any duplicate rows in the dataframe.

Decide on a strategy for handling the duplicates. Options include:
- Dropping all duplicate rows
- Keeping only the first occurrence of each duplicated row
- Keeping only the last occurrence of each duplicated row
- Dropping duplicates based on a subset of columns
- Dropping duplicates based on a specific column

Implement your chosen strategy using the `drop_duplicates()` function.

Verify that your strategy has successfully handled the duplicates by checking for duplicates again using `.duplicated()`.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling duplicates.

Save the cleaned dataset to a new CSV file.

*Hint*: *after dropping duplicates, reset the index to ensure consistency*.

### Solution — remove duplicates and export

`duplicated().sum()` counts repeated complete rows. We keep the first occurrence because identical rows contain no information for choosing one copy over another. `reset_index(drop=True)` creates a clean consecutive index, and `to_csv(..., index=False)` avoids writing that pandas index as an unwanted CSV column.


In [ ]:
duplicates_before = df.duplicated().sum()
print(f"Duplicate rows before cleaning: {duplicates_before}")

df = df.drop_duplicates(keep="first").reset_index(drop=True)

duplicates_after = df.duplicated().sum()
print(f"Duplicate rows after cleaning: {duplicates_after}")
print(f"Final shape: {df.shape}")

output_file = "cleaned_customer_data.csv"
df.to_csv(output_file, index=False)
print(f"Saved: {output_file}")


# Bonus: Challenge 2: creating functions on a separate `py` file

Put all the data cleaning and formatting steps into functions, and create a main function that performs all the cleaning and formatting.

Write these functions in separate .py file(s). By putting these steps into functions, we can make the code more modular and easier to maintain.

*Hint: autoreload module is a utility module in Python that allows you to automatically reload modules in the current session when changes are made to the source code. This can be useful in situations where you are actively developing code and want to see the effects of changes you make without having to constantly restart the Python interpreter or Jupyter Notebook kernel.*

### Bonus solution — reusable functions

The file `cleaning_functions.py` contains one function per cleaning stage plus `clean_data()`, which runs the complete pipeline. `%autoreload 2` tells Jupyter to reload the module automatically after its source changes.

We rerun the pipeline from the raw dataset—not from the already-cleaned `df`—to demonstrate that the module works independently.


In [ ]:
%load_ext autoreload
%autoreload 2

from cleaning_functions import clean_data

raw_df = pd.read_csv(data_url)
cleaned_with_functions = clean_data(raw_df)

display(cleaned_with_functions.head())
print("Shape:", cleaned_with_functions.shape)
print("Missing values:", cleaned_with_functions.isna().sum().sum())
print("Duplicate rows:", cleaned_with_functions.duplicated().sum())


# Bonus: Challenge 3: Analyzing Clean and Formated Data

You have been tasked with analyzing the data to identify potential areas for improving customer retention and profitability. Your goal is to identify customers with a high policy claim amount and a low customer lifetime value.

In the Pandas Lab, we only looked at high policy claim amounts because we couldn't look into low customer lifetime values. If we had tried to work with that column, we wouldn't have been able to because customer lifetime value wasn't clean and in its proper format. So after cleaning and formatting the data, let's get some more interesting insights!

Instructions:

- Review the statistics again for total claim amount and customer lifetime value to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount and a low customer lifetime value. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount, and clients with a low customer lifetime value to be those in the bottom 25% of the customer lifetime value. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile and a customer lifetime value in the bottom 25th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount and low customer lifetime value data. To do so, select both columns of the dataframe simultaneously and pass it to the `.describe()` method. This will give you descriptive statistics, such as mean, median, standard deviation, minimum and maximum values for both columns at the same time, allowing you to compare and analyze their characteristics.

### Bonus solution — target high-claim, low-CLV customers

`describe()` reviews both measures. The 75th percentile of total claims is the high-claim threshold, while the 25th percentile of customer lifetime value is the low-CLV threshold.

The Boolean conditions are joined with `&`, meaning a row must meet **both** requirements. Parentheses around each condition are required in pandas.


In [ ]:
analysis_columns = ["total_claim_amount", "customer_lifetime_value"]
display(df[analysis_columns].describe())

claim_q75 = df["total_claim_amount"].quantile(0.75)
clv_q25 = df["customer_lifetime_value"].quantile(0.25)

high_claim_low_clv = df.loc[
    (df["total_claim_amount"] > claim_q75)
    & (df["customer_lifetime_value"] < clv_q25)
].copy()

print(f"High-claim threshold (75th percentile): {claim_q75:,.2f}")
print(f"Low-CLV threshold (25th percentile): {clv_q25:,.2f}")
print(f"Customers meeting both conditions: {len(high_claim_low_clv)}")

display(high_claim_low_clv.head())
display(high_claim_low_clv[analysis_columns].describe())


**Interpretation:** This subset represents customers whose claims are unusually high relative to the whole dataset while their estimated lifetime value is unusually low. They may warrant closer profitability and retention review. The filter identifies a segment for investigation; it does not by itself prove that any individual customer is unprofitable.
